**Importation des bibliothèques**

In [1]:
from sentence_transformers import SentenceTransformer
import tensorflow_datasets as tfds
import numpy as np
import pandas as pd
import ast
from sklearn.metrics.pairwise import cosine_similarity
from ast import literal_eval

/usr/local/lib/python3.10/dist-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Chargement et préparation des données**

In [3]:
# Charger les datasets
queries = pd.read_csv('/content/drive/MyDrive/queries.csv')
corpus = pd.read_csv('/content/drive/MyDrive/corp.csv')
test = pd.read_csv('/content/drive/MyDrive/relevance_data_test.csv')

In [4]:
queries.head()

,_id,text,query,narrative
0,1,what is the origin of COVID-19,coronavirus origin,seeking range of information about the SARS-Co...
1,2,how does the coronavirus respond to changes in...,coronavirus response to weather changes,seeking range of information about the SARS-Co...
2,3,will SARS-CoV2 infected people develop immunit...,coronavirus immunity,seeking studies of immunity developed due to i...
3,4,what causes death from Covid-19?,how do people die from the coronavirus,Studies looking at mechanisms of death from Co...
4,5,what drugs have been active against SARS-CoV o...,animal models of COVID-19,Papers that describe the results of testing d...


In [5]:
corpus.head()

,_id,title,text,metadata
0,ug7v899j,Clinical features of culture-proven Mycoplasma...,OBJECTIVE: This retrospective chart review des...,{'url': 'https://www.ncbi.nlm.nih.gov/pmc/arti...
1,02tnwd4m,Nitric oxide: a pro-inflammatory mediator in l...,Inflammatory diseases of the respiratory tract...,{'url': 'https://www.ncbi.nlm.nih.gov/pmc/arti...
2,ejv2xln0,Surfactant protein-D and pulmonary host defense,Surfactant protein-D (SP-D) participates in th...,{'url': 'https://www.ncbi.nlm.nih.gov/pmc/arti...
3,2b73a28n,Role of endothelin-1 in lung disease,Endothelin-1 (ET-1) is a 21 amino acid peptide...,{'url': 'https://www.ncbi.nlm.nih.gov/pmc/arti...
4,9785vg6d,Gene expression in epithelial cells in respons...,Respiratory syncytial virus (RSV) and pneumoni...,{'url': 'https://www.ncbi.nlm.nih.gov/pmc/arti...


In [6]:
test

,topic_id,round_id,cord_uid,relevancy
0,1,4.5,005b2j4b,2
1,1,4.0,00fmeepz,1
2,1,0.5,010vptx3,2
3,1,2.5,0194oljo,1
4,1,4.0,021q9884,1
...,...,...,...,...
69311,50,5.0,zvop8bxh,2
69312,50,5.0,zwf26o63,1
69313,50,5.0,zwsvlnwe,0
69314,50,5.0,zxr01yln,1


**Chargement du modèle pour les embeddings**

In [7]:
# Charger le modèle pour les embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [8]:
# Créer une nouvelle colonne 'combined' dans 'queries' qui concatène 'query' et 'text'
queries['query_text'] = queries['query'] + ' ' + corpus['text']

# Calculer les embeddings pour la nouvelle colonne
combined_embeddings = model.encode(queries['query_text'].tolist())

# Ajouter les embeddings comme nouvelle colonne dans 'queries'
queries['query_embedding'] = list(combined_embeddings)

queries.head()

,_id,text,query,narrative,query_text,query_embedding
0,1,what is the origin of COVID-19,coronavirus origin,seeking range of information about the SARS-Co...,coronavirus origin OBJECTIVE: This retrospecti...,"[0.06701338, 0.048250888, -0.056257073, 0.0157..."
1,2,how does the coronavirus respond to changes in...,coronavirus response to weather changes,seeking range of information about the SARS-Co...,coronavirus response to weather changes Inflam...,"[0.038665265, 0.015554365, -0.014207781, 0.055..."
2,3,will SARS-CoV2 infected people develop immunit...,coronavirus immunity,seeking studies of immunity developed due to i...,coronavirus immunity Surfactant protein-D (SP-...,"[-0.046173822, -0.06418642, -0.012989899, -0.0..."
3,4,what causes death from Covid-19?,how do people die from the coronavirus,Studies looking at mechanisms of death from Co...,how do people die from the coronavirus Endothe...,"[0.029778315, -0.027963227, -0.021062018, 0.00..."
4,5,what drugs have been active against SARS-CoV o...,animal models of COVID-19,Papers that describe the results of testing d...,animal models of COVID-19 Respiratory syncytia...,"[-0.072889924, -0.06498722, -0.07472659, -0.05..."


In [9]:
corpus['text'] = corpus['text'].astype(str)

# Calculer les embeddings pour la colonne 'text' en utilisant le modèle (ici model doit être défini)
doc_embeddings = model.encode(corpus['text'].tolist(), show_progress_bar=True)

# Ajouter les embeddings comme nouvelle colonne 'doc_embedding'
corpus['doc_embedding'] = doc_embeddings.tolist()

# Afficher les premières lignes du DataFrame
corpus.head()

Batches:   0%|          | 0/5355 [00:00<?, ?it/s]

,_id,title,text,metadata,doc_embedding
0,ug7v899j,Clinical features of culture-proven Mycoplasma...,OBJECTIVE: This retrospective chart review des...,{'url': 'https://www.ncbi.nlm.nih.gov/pmc/arti...,"[0.053586695343256, 0.03623261675238609, -0.05..."
1,02tnwd4m,Nitric oxide: a pro-inflammatory mediator in l...,Inflammatory diseases of the respiratory tract...,{'url': 'https://www.ncbi.nlm.nih.gov/pmc/arti...,"[0.03221406415104866, 0.0010244973236694932, -..."
2,ejv2xln0,Surfactant protein-D and pulmonary host defense,Surfactant protein-D (SP-D) participates in th...,{'url': 'https://www.ncbi.nlm.nih.gov/pmc/arti...,"[-0.06293858587741852, -0.09024284780025482, 0..."
3,2b73a28n,Role of endothelin-1 in lung disease,Endothelin-1 (ET-1) is a 21 amino acid peptide...,{'url': 'https://www.ncbi.nlm.nih.gov/pmc/arti...,"[0.041389573365449905, -0.03688059374690056, -..."
4,9785vg6d,Gene expression in epithelial cells in respons...,Respiratory syncytial virus (RSV) and pneumoni...,{'url': 'https://www.ncbi.nlm.nih.gov/pmc/arti...,"[-0.08775756508111954, -0.05350399389863014, -..."


**Calcul de la similarité cosinus et sélection des 30 documents les plus pertinents pour chaque requête**

In [10]:
# Conversion des embeddings en numpy arrays
corpus_embeddings = np.array(corpus['doc_embedding'].tolist())
queries_embeddings = np.array(queries['query_embedding'].tolist())

# Calcul de la similarité cosinus
cos_sim_matrix = cosine_similarity(queries_embeddings, corpus_embeddings)

# Liste pour stocker les résultats
results = []

# Sélectionner les 30 documents les plus pertinents pour chaque requête
for i in range(cos_sim_matrix.shape[0]):
    # Trier les indices des documents en fonction de la similarité pour la requête i
    relevant_docs_indices = np.argsort(cos_sim_matrix[i])[::-1]  # Tri décroissant

    # Sélectionner seulement les 30 premiers documents
    top_30_indices = relevant_docs_indices[:30]

    # Pour chaque document sélectionné, récupérer le titre, le texte, l'ID et la similarité
    for doc_idx in top_30_indices:
        document = corpus.iloc[doc_idx]  # Document sélectionné
        query = queries.iloc[i]  # Requête correspondante

        # Ajouter les informations dans la liste des résultats
        results.append({
            'query_id': query['_id'],
            'query_text': query['query_text'],
            'document_id': document['_id'],
            'document_text': document['text'],
            'cosine_similarity': cos_sim_matrix[i][doc_idx]  # Ajouter la similarité
        })

# Créer un DataFrame avec les résultats
retrieved_docs_top_30 = pd.DataFrame(results)

# Enregistrer les résultats dans un fichier CSV avec un nom indiquant les 30 premiers documents
retrieved_docs_top_30.to_csv('/content/drive/MyDrive/retrieved_documents_top_30.csv', index=False)

# Afficher le DataFrame avec les résultats
retrieved_docs_top_30

,query_id,query_text,document_id,document_text,cosine_similarity
0,1,coronavirus origin OBJECTIVE: This retrospecti...,ug7v899j,OBJECTIVE: This retrospective chart review des...,0.904804
1,1,coronavirus origin OBJECTIVE: This retrospecti...,gv1gfypn,BACKGROUND: Mycoplasma pneumoniae is a common ...,0.805166
2,1,coronavirus origin OBJECTIVE: This retrospecti...,ozto1jkd,We evaluated the microbiological diagnosis in ...,0.804871
3,1,coronavirus origin OBJECTIVE: This retrospecti...,mt4dftwj,Coronavirus disease 2019 (COVID‐19) caused by ...,0.774033
4,1,coronavirus origin OBJECTIVE: This retrospecti...,8z92r3k9,Coronavirus disease 2019 (COVID-19) caused by ...,0.767901
...,...,...,...,...,...
1495,50,mRNA vaccine coronavirus BACKGROUND: As a numb...,1a8fm18m,"The rapid emergence of a highly pathogenic, re...",0.672494
1496,50,mRNA vaccine coronavirus BACKGROUND: As a numb...,syt4r964,Coronaviruses (CoVs) are a large family of env...,0.670795
1497,50,mRNA vaccine coronavirus BACKGROUND: As a numb...,ljggnelt,"Over the past several decades, we have witness...",0.670734
1498,50,mRNA vaccine coronavirus BACKGROUND: As a numb...,9psmkf55,New viral respiratory pathogens are emerging w...,0.669821


**Filtrage des documents Pertinents**

In [17]:
# Lire le fichier CSV dans un DataFrame
retrieved_docs = pd.read_csv('/content/drive/MyDrive/retrieved_documents_top_30.csv')

# Accéder à la colonne 'cosine_similarity'
cos_similarity_matrix = retrieved_docs['cosine_similarity'].values  # Transformer en tableau numpy

# Filtrer les documents pertinents dans le dataset de test
test_relevant_docs = test[test['relevancy'].isin([1, 2])]

# Créer une liste pour stocker les résultats pertinents
pertinent_results = []

# Parcourir chaque ligne de retrieved_docs (chaque document récupéré)
for i, row in retrieved_docs.iterrows():
    # Récupérer l'ID du document et la similarité cosinus
    document_id = row['document_id']
    cosine_similarity = row['cosine_similarity']

    # Vérifier si le document est pertinent dans le test set
    if document_id in test_relevant_docs['cord_uid'].values:
        # Trouver la requête correspondante
        query_id = row['query_id']
        query_text = row['query_text']
        document_text = row['document_text']

        # Ajouter le document pertinent à la liste des résultats
        pertinent_results.append({
            'query_id': query_id,
            'query_text': query_text,
            'document_id': document_id,
            'document_text': document_text,
            'cosine_similarity': cosine_similarity
        })

# Convertir la liste des résultats pertinents en DataFrame
pertinent_resultats = pd.DataFrame(pertinent_results)

# Afficher le DataFrame des résultats pertinents
pertinent_resultats

,query_id,query_text,document_id,document_text,cosine_similarity
0,1,coronavirus origin OBJECTIVE: This retrospecti...,coftgksc,OBJECTIVE: To study the clinical features of a...,0.722037
1,1,coronavirus origin OBJECTIVE: This retrospecti...,3vwehyud,OBJECTIVE To study the clinical features of as...,0.713714
2,2,coronavirus response to weather changes Inflam...,hs2q61gw,Pandemic coronavirus disease 2019 (COVID-19) i...,0.656287
3,2,coronavirus response to weather changes Inflam...,c8mmlyjn,(1) Background: The emergence of severe acute ...,0.640775
4,2,coronavirus response to weather changes Inflam...,x74cq2k5,Abstract The emergence of viral respiratory pa...,0.637997
...,...,...,...,...,...
256,50,mRNA vaccine coronavirus BACKGROUND: As a numb...,jxr0phr1,Abstract The global epidemic of severe acute r...,0.681288
257,50,mRNA vaccine coronavirus BACKGROUND: As a numb...,uw8qxw74,"Abstract In the recent two decades, three glob...",0.678722
258,50,mRNA vaccine coronavirus BACKGROUND: As a numb...,1a8fm18m,"The rapid emergence of a highly pathogenic, re...",0.672494
259,50,mRNA vaccine coronavirus BACKGROUND: As a numb...,syt4r964,Coronaviruses (CoVs) are a large family of env...,0.670795


**Création des pertinences binaires pour chaque requête**

In [18]:
# Créer un dictionnaire pour stocker le nombre de documents pertinents par query_id
relevant_retrieved_in_top30 = {}

# Créer un dictionnaire pour stocker les pertinences binaires (0 ou 1) pour chaque query_id
relevant_binary_dict = {}

# Parcourir chaque ligne de retrieved_docs (chaque document récupéré)
for i, row in retrieved_docs.iterrows():
    document_id = row['document_id']
    query_id = row['query_id']

    # Vérifier si le document est pertinent dans le dataset de test
    if document_id in test_relevant_docs['cord_uid'].values:
        # Si le query_id n'existe pas encore dans le dictionnaire, l'initialiser
        if query_id not in relevant_retrieved_in_top30:
            relevant_retrieved_in_top30[query_id] = 0

        # Incrémenter le compteur de documents pertinents pour cette requête
        relevant_retrieved_in_top30[query_id] += 1

# Afficher le dictionnaire des documents pertinents par query_id
print("Documents pertinents par query_id :")
print(relevant_retrieved_in_top30)

# Parcourir les résultats récupérés pour chaque query_id
for query_id in relevant_retrieved_in_top30:
    # Filtrer les documents pour cette query_id
    query_docs = retrieved_docs[retrieved_docs['query_id'] == query_id]

    # Créer une liste pour stocker les pertinences binaires
    binary_relevance = []

    # Parcourir les documents récupérés et vérifier si ils sont pertinents
    for idx, row in query_docs.iterrows():
        document_id = row['document_id']

        # Si le document est pertinent, ajouter 1, sinon ajouter 0
        if document_id in test_relevant_docs['cord_uid'].values:
            binary_relevance.append(1)
        else:
            binary_relevance.append(0)

    # Ajouter la liste de pertinence binaire au dictionnaire
    relevant_binary_dict[query_id] = binary_relevance

# Afficher le dictionnaire des pertinences binaires
print("\nPertinences binaires par query_id :")
print(relevant_binary_dict)


Documents pertinents par query_id :
{1: 2, 2: 6, 3: 5, 4: 8, 5: 3, 7: 2, 8: 7, 9: 11, 10: 3, 13: 1, 14: 1, 16: 4, 17: 4, 18: 9, 19: 3, 20: 1, 21: 4, 22: 2, 23: 19, 24: 1, 25: 2, 26: 8, 27: 4, 28: 1, 29: 6, 30: 5, 31: 1, 32: 7, 33: 8, 34: 2, 35: 1, 36: 2, 37: 9, 39: 8, 40: 19, 42: 27, 43: 1, 44: 23, 45: 9, 46: 11, 48: 1, 49: 1, 50: 9}

Pertinences binaires par query_id :
{1: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1], 2: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0], 3: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], 4: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0], 5: [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 7: [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 8: [0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1

In [19]:
relevant_docs = test[test['relevancy'] == 1].groupby('topic_id').size().to_dict()

# Afficher les résultats
print(relevant_docs)

{1: 362, 2: 71, 3: 443, 4: 331, 5: 339, 6: 328, 7: 50, 8: 391, 9: 104, 10: 203, 11: 226, 12: 295, 13: 656, 14: 172, 15: 266, 16: 236, 17: 372, 18: 319, 19: 68, 20: 288, 21: 80, 22: 216, 23: 194, 24: 150, 25: 167, 26: 148, 27: 580, 28: 74, 29: 275, 30: 211, 31: 213, 32: 80, 33: 125, 34: 74, 35: 32, 36: 105, 37: 144, 38: 618, 39: 438, 40: 217, 41: 87, 42: 23, 43: 97, 44: 182, 45: 352, 46: 109, 47: 113, 48: 202, 49: 131, 50: 98}


**Calcul des métriques de performance pour l'évaluation des résultats de recherche (P@k, R@k, F1@k, AP et MAP)**

In [20]:
# Fonction pour calculer P@k (Précision à k)
def precision_at_k(relevant_binary_dict, k=30):
    p_at_k = {}
    for query_id, relevant_list in relevant_binary_dict.items():
        # Compter le nombre de documents pertinents dans les k premiers
        relevant_count = sum(relevant_list[:k])  # Nombre de 1s dans les k premiers documents
        p_at_k[query_id] = relevant_count / k  # Diviser par k pour obtenir la précision à k
    return p_at_k

# Fonction pour calculer R@k (Rappel à k)
def recall_at_k(relevant_binary_dict, total_relevant_docs, k=30):
    r_at_k = {}
    for query_id, relevant_list in relevant_binary_dict.items():
        # Nombre de documents pertinents récupérés dans les k premiers
        relevant_retrieved = sum(relevant_list[:k])  # Nombre de 1s dans les k premiers documents
        total_relevant = total_relevant_docs.get(query_id, 0)  # Nombre total de documents pertinents pour cette requête
        r_at_k[query_id] = relevant_retrieved / total_relevant if total_relevant > 0 else 0  # Calcul du rappel
    return r_at_k

# Fonction pour calculer F1@k (Score F1 à k)
def f1_at_k(p_at_k, r_at_k):
    f1_at_k = {}
    for query_id in p_at_k:
        precision = p_at_k.get(query_id, 0)
        recall = r_at_k.get(query_id, 0)
        # Calcul du score F1
        f1_at_k[query_id] = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return f1_at_k

# Fonction pour calculer l'Average Precision (AP) pour chaque requête
def average_precision(relevant_retrieved_list):
    relevant_count = 0
    precision_sum = 0

    for i, relevance in enumerate(relevant_retrieved_list):
        if relevance == 1:
            relevant_count += 1
            precision_sum += relevant_count / (i + 1)  # Précision à la position i+1

    # Retourner l'AP, si des documents pertinents sont présents, sinon retourner 0
    return precision_sum / relevant_count if relevant_count > 0 else 0

# Fonction pour calculer la MAP (Mean Average Precision)
def mean_average_precision(relevant_retrieved_in_top_k):
    ap_list = [average_precision(relevant_retrieved_list) for relevant_retrieved_list in relevant_retrieved_in_top_k.values()]
    # Retourner la moyenne des AP pour obtenir le MAP
    return sum(ap_list) / len(ap_list) if ap_list else 0

In [21]:
# Calculer P@k
p_at_k = precision_at_k(relevant_binary_dict)

# Calculer R@k
r_at_k = recall_at_k(relevant_binary_dict, relevant_docs)

# Calculer F1@k
f1_at_k_score = f1_at_k(p_at_k, r_at_k)

# Calculer l'Average Precision (AP) pour chaque requête
ap_at_k = {query_id: average_precision(relevant_list) for query_id, relevant_list in relevant_binary_dict.items()}

# Calculer la MAP
map_score = mean_average_precision(relevant_binary_dict)

# Créer un DataFrame avec les résultats
df_results = pd.DataFrame({
    'query_id': list(p_at_k.keys()),
    'P@30': list(p_at_k.values()),
    'R@30': list(r_at_k.values()),
    'F1@30': list(f1_at_k_score.values()),
    'Average Precision': list(ap_at_k.values())
})

# Afficher le DataFrame
df_results

,query_id,P@30,R@30,F1@30,Average Precision
0,1,0.066667,0.005525,0.010204,0.054167
1,2,0.200000,0.084507,0.118812,0.167304
2,3,0.166667,0.011287,0.021142,0.154682
3,4,0.266667,0.024169,0.044321,0.202946
4,5,0.100000,0.008850,0.016260,0.151323
5,7,0.066667,0.040000,0.050000,0.105556
6,8,0.233333,0.017903,0.033254,0.263961
7,9,0.366667,0.105769,0.164179,0.490882
8,10,0.100000,0.014778,0.025751,0.111899
9,13,0.033333,0.001524,0.002915,0.071429


In [22]:
print(f"MAP Score: {map_score}")

MAP Score: 0.23270312590708733


In [23]:
df_results.to_csv('/content/drive/MyDrive/metrics.csv', index=False)